# Exploração da tabela de consumo

Esta análise valida a tabela antes do baseline: distribuição dos alvos, concentração por canal e corte temporal de avaliação.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TABLE_PATH = ROOT / 'data' / 'gold' / 'pilot_consumption_table.csv'

data = pd.read_csv(TABLE_PATH, parse_dates=['published_at'])
data.shape, data['channel_id'].nunique()

In [ ]:
required_columns = {
    'video_id', 'channel_id', 'published_at', 'baseline_log_likes',
    'target_log_likes', 'target_relative_to_prior_median', 'split'
}
assert required_columns.issubset(data.columns)
assert data['video_id'].is_unique
assert data['split'].isin(['train', 'test']).all()

feature_columns = [
    'duration_seconds', 'channel_prior_posts', 'channel_prior_median_likes',
    'days_since_previous_post', 'publish_hour_utc', 'publish_weekday',
    'title_length', 'description_length', 'tag_count', 'baseline_log_likes',
]
data[feature_columns + ['target_log_likes', 'target_relative_to_prior_median']].isna().sum()

## Distribuição dos alvos

O primeiro modelo usará `target_log_likes`, porque a escala de curtidas é muito assimétrica. O alvo relativo continua disponível para experimento posterior.

In [ ]:
quantiles = data[['target_log_likes', 'target_relative_to_prior_median']].quantile(
    [0, .25, .5, .75, .9, .95, 1]
).round(3)
display(quantiles)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(data['target_log_likes'], bins=40, color='#386cb0', edgecolor='white')
axes[0].set(title='Alvo principal: log(1 + curtidas)', xlabel='target_log_likes', ylabel='Shorts')

relative_cap = data['target_relative_to_prior_median'].quantile(.99)
axes[1].hist(
    data['target_relative_to_prior_median'].clip(upper=relative_cap),
    bins=40, color='#7fc97f', edgecolor='white'
)
axes[1].set(
    title='Alvo relativo ao histórico (truncado no p99)',
    xlabel='curtidas / mediana anterior', ylabel='Shorts'
)
plt.tight_layout()

## Desbalanceamento por canal

A análise é feita por canal, pois vídeos do mesmo canal compartilham audiência e frequência de publicação.

In [ ]:
channel_summary = (
    data.groupby(['channel_name', 'sector', 'channel_type'], as_index=False)
    .agg(
        shorts=('video_id', 'size'),
        share=('video_id', lambda values: len(values) / len(data)),
        median_likes=('target_log_likes', lambda values: np.expm1(values).median()),
        train=('split', lambda values: (values == 'train').sum()),
        test=('split', lambda values: (values == 'test').sum()),
    )
    .sort_values('shorts', ascending=False)
)
channel_summary_display = channel_summary.copy()
channel_summary_display['share'] = channel_summary_display['share'].map(lambda value: f'{value:.1%}')
channel_summary_display['median_likes'] = channel_summary_display['median_likes'].map(lambda value: f'{value:,.0f}')
display(channel_summary_display)

plot_data = channel_summary.sort_values('share')
ax = plot_data.plot.barh(x='channel_name', y='share', legend=False, figsize=(9, 6), color='#fdc086')
ax.axvline(.15, linestyle='--', color='#555555', label='limite de referência: 15%')
ax.set(xlabel='Participação na tabela', ylabel='', title='Participação de cada canal')
ax.xaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
ax.legend()
plt.tight_layout()

## Corte temporal formal

O teste contém os 20% mais recentes de **cada** canal. Isso mantém todos os canais no treino e no teste, evita avaliar o futuro com informação futura e produz uma comparação justa com o baseline de mediana histórica.

In [ ]:
temporal_audit = (
    data.groupby('channel_name')
    .apply(lambda channel: pd.Series({
        'train_rows': channel['split'].eq('train').sum(),
        'test_rows': channel['split'].eq('test').sum(),
        'last_train': channel.loc[channel['split'].eq('train'), 'published_at'].max(),
        'first_test': channel.loc[channel['split'].eq('test'), 'published_at'].min(),
    }), include_groups=False)
    .sort_values('test_rows', ascending=False)
)
temporal_audit['ordered'] = temporal_audit['last_train'] < temporal_audit['first_test']
assert temporal_audit['ordered'].all()
display(temporal_audit)
print(data['split'].value_counts().rename('rows'))
print(f"Teste: {data['split'].eq('test').mean():.1%} da tabela")

## Decisões para o baseline

- Treinar apenas com linhas `split == 'train'` (1.467 linhas).
- Avaliar uma única vez nas 375 linhas de `split == 'test'`.
- Comparar o modelo global multicanal com `baseline_log_likes`, a mediana de curtidas dos posts anteriores de cada canal.
- Usar `target_log_likes` como alvo inicial; o alvo relativo fica reservado para comparação posterior.